# Data Analysis — Input Text Corpus
**Project:** Text-to-Video Animated Pipeline
**Notebook:** `02_data_analysis.ipynb`

## 1. Setup & Data Loading

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import Counter

matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 130,
})

PALETTE = {'education': '#4C72B0', 'product': '#55A868', 'tutorial': '#C44E52'}

META_PATH  = '../data/metadata.csv'
TEXTS_DIR  = '../data/input_texts'

df = pd.read_csv(META_PATH)
df.head(10)

## 2. Feature Extraction from Raw Text Files

In [ ]:
def extract_features(file_name, texts_dir):
    path = os.path.join(texts_dir, file_name)
    with open(path, 'r', encoding='utf-8') as f:
        raw = f.read()
    text = raw.strip()
    words  = text.split()
    sents  = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    chars  = len(text)
    paras  = [p.strip() for p in text.split('\n\n') if p.strip()]
    avg_sent_len = round(len(words) / max(len(sents), 1), 1)
    return {
        'char_count':    chars,
        'sentence_count': len(sents),
        'paragraph_count': len(paras),
        'avg_words_per_sentence': avg_sent_len,
        'unique_words': len(set(w.lower().strip('.,;:!?') for w in words)),
    }

features = df['file_name'].apply(lambda fn: pd.Series(extract_features(fn, TEXTS_DIR)))
df = pd.concat([df, features], axis=1)
df[['file_name','category','word_count','char_count','sentence_count',
    'paragraph_count','avg_words_per_sentence','unique_words','is_synthetic']]

## 3. Descriptive Statistics

In [ ]:
numeric_cols = ['word_count','char_count','sentence_count',
                'paragraph_count','avg_words_per_sentence','unique_words']

print('=== Overall Statistics ===')
print(df[numeric_cols].describe().round(2).to_string())
print()
print('=== Statistics by Category ===')
print(df.groupby('category')[numeric_cols].mean().round(2).to_string())
print()
print('=== Source breakdown ===')
print(df['is_synthetic'].value_counts().rename({True:'synthetic', False:'real'}).to_string())

## 4. Visualizations

### 4.1 Word-count Distribution by Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Word-Count Distribution by Category', fontsize=14, fontweight='bold', y=1.01)

ax = axes[0]
categories = df['category'].unique()
for i, cat in enumerate(sorted(categories)):
    vals = df.loc[df['category'] == cat, 'word_count'].values
    ax.scatter([i]*len(vals), vals, color=PALETTE[cat], s=90, zorder=3,
               edgecolors='white', linewidths=0.6)
    bp = ax.boxplot(vals, positions=[i], widths=0.35, patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[cat], alpha=0.25),
                    medianprops=dict(color=PALETTE[cat], linewidth=2),
                    whiskerprops=dict(color=PALETTE[cat]),
                    capprops=dict(color=PALETTE[cat]),
                    flierprops=dict(marker=''))
ax.set_xticks(range(len(sorted(categories))))
ax.set_xticklabels([c.capitalize() for c in sorted(categories)])
ax.set_ylabel('Word Count')
ax.set_title('Box + Strip Plot')

ax2 = axes[1]
cats_sorted = sorted(categories)
means  = [df.loc[df['category']==c,'word_count'].mean() for c in cats_sorted]
stds   = [df.loc[df['category']==c,'word_count'].std()  for c in cats_sorted]
colors = [PALETTE[c] for c in cats_sorted]
bars   = ax2.bar(cats_sorted, means, color=colors, edgecolor='white',
                 linewidth=1.2, width=0.5, zorder=2)
ax2.errorbar(cats_sorted, means, yerr=stds, fmt='none',
             ecolor='#333333', capsize=5, linewidth=1.5, zorder=3)
for bar, val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
             f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_xticklabels([c.capitalize() for c in cats_sorted])
ax2.set_ylabel('Mean Word Count')
ax2.set_title('Mean ± Std Dev')

plt.tight_layout()
plt.savefig('../docs/figures/fig1_word_count_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → docs/figures/fig1_word_count_distribution.png')

### 4.2 Corpus Composition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Corpus Composition', fontsize=14, fontweight='bold')

ax = axes[0]
cat_counts = df['category'].value_counts()
wedge_colors = [PALETTE[c] for c in cat_counts.index]
wedges, texts, autotexts = ax.pie(
    cat_counts.values, labels=None,
    colors=wedge_colors, autopct='%1.0f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax.legend(
    [f'{c.capitalize()} ({v})' for c, v in zip(cat_counts.index, cat_counts.values)],
    loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=9
)
ax.set_title('By Category')

ax2 = axes[1]
synth_table = df.groupby(['category','is_synthetic']).size().unstack(fill_value=0)
synth_table.columns = ['Real', 'Synthetic']
cats = [c for c in sorted(synth_table.index)]
real_vals  = [synth_table.loc[c,'Real']      if c in synth_table.index else 0 for c in cats]
synth_vals = [synth_table.loc[c,'Synthetic'] if c in synth_table.index else 0 for c in cats]
x = range(len(cats))
bars_r = ax2.bar(x, real_vals,  label='Real',      color=[PALETTE[c] for c in cats], width=0.5)
bars_s = ax2.bar(x, synth_vals, bottom=real_vals,  label='Synthetic',
                 color=[PALETTE[c] for c in cats], width=0.5, alpha=0.45,
                 hatch='//', edgecolor='white')
ax2.set_xticks(list(x))
ax2.set_xticklabels([c.capitalize() for c in cats])
ax2.set_ylabel('Number of Texts')
ax2.set_title('Real vs. AI-Generated per Category')
real_patch  = mpatches.Patch(color='grey', label='Real')
synth_patch = mpatches.Patch(facecolor='grey', alpha=0.45, hatch='//', label='AI-Generated')
ax2.legend(handles=[real_patch, synth_patch], fontsize=9)

plt.tight_layout()
plt.savefig('../docs/figures/fig2_corpus_composition.png', bbox_inches='tight')
plt.show()
print('Saved → docs/figures/fig2_corpus_composition.png')

### 4.3 Text Complexity: Sentences & Vocabulary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Text Complexity Indicators', fontsize=14, fontweight='bold')

ax = axes[0]
for i, (_, row) in enumerate(df.iterrows()):
    ax.scatter(i, row['avg_words_per_sentence'],
               color=PALETTE[row['category']], s=80, zorder=3,
               edgecolors='white', linewidths=0.5)
ax.axhline(df['avg_words_per_sentence'].mean(), color='#333333',
           linestyle='--', linewidth=1.2, label=f"Mean = {df['avg_words_per_sentence'].mean():.1f}")
ax.set_xticks(range(len(df)))
ax.set_xticklabels(df['file_name'].str.replace('.txt','',regex=False),
                   rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('Avg Words per Sentence')
ax.set_title('Sentence Complexity per Document')
patches = [mpatches.Patch(color=PALETTE[c], label=c.capitalize()) for c in PALETTE]
ax.legend(handles=patches + [plt.Line2D([0],[0],color='#333333',linestyle='--',label='Mean')],
          fontsize=8)

ax2 = axes[1]
for _, row in df.iterrows():
    ax2.scatter(row['word_count'], row['unique_words'],
                color=PALETTE[row['category']], s=90, zorder=3,
                edgecolors='white', linewidths=0.5)
    ax2.annotate(row['file_name'].replace('.txt',''),
                 (row['word_count'], row['unique_words']),
                 textcoords='offset points', xytext=(5,3), fontsize=6.5, color='#444')
z = np.polyfit(df['word_count'], df['unique_words'], 1)
p = np.poly1d(z)
xs = np.linspace(df['word_count'].min(), df['word_count'].max(), 100)
ax2.plot(xs, p(xs), color='#888', linestyle='--', linewidth=1.2, label='Trend')
ax2.set_xlabel('Total Word Count')
ax2.set_ylabel('Unique Word Count')
ax2.set_title('Vocabulary Richness')
patches2 = [mpatches.Patch(color=PALETTE[c], label=c.capitalize()) for c in PALETTE]
ax2.legend(handles=patches2 + [plt.Line2D([0],[0],color='#888',linestyle='--',label='Trend')],
           fontsize=8)

plt.tight_layout()
plt.savefig('../docs/figures/fig3_text_complexity.png', bbox_inches='tight')
plt.show()
print('Saved → docs/figures/fig3_text_complexity.png')

### 4.4 Estimated Scene Coverage

In [ ]:
WORDS_PER_SCENE_LOW  = 25
WORDS_PER_SCENE_HIGH = 35

df['scenes_low']  = (df['word_count'] / WORDS_PER_SCENE_HIGH).apply(np.floor).astype(int).clip(lower=1)
df['scenes_high'] = (df['word_count'] / WORDS_PER_SCENE_LOW ).apply(np.ceil ).astype(int)
df['scenes_mid']  = ((df['scenes_low'] + df['scenes_high']) / 2).round(1)

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle('Estimated Scene Count per Document\n(pipeline target: 5–8 scenes)',
             fontsize=13, fontweight='bold')

indices = range(len(df))
bar_colors = [PALETTE[c] for c in df['category']]

ax.bar(indices, df['scenes_high'] - df['scenes_low'],
       bottom=df['scenes_low'], color=bar_colors,
       alpha=0.3, width=0.6, label='Estimated range')
ax.scatter(indices, df['scenes_mid'],
           color=bar_colors, s=60, zorder=4, edgecolors='white')

ax.axhspan(5, 8, color='#2ca02c', alpha=0.08, label='Target zone (5–8)')
ax.axhline(5, color='#2ca02c', linewidth=1,   linestyle=':')
ax.axhline(8, color='#2ca02c', linewidth=1,   linestyle=':')

ax.set_xticks(list(indices))
ax.set_xticklabels(df['file_name'].str.replace('.txt','',regex=False),
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Number of Scenes')
patches = [mpatches.Patch(color=PALETTE[c], label=c.capitalize()) for c in PALETTE]
green_band = mpatches.Patch(color='#2ca02c', alpha=0.2, label='Pipeline target (5–8)')
ax.legend(handles=patches + [green_band], fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('../docs/figures/fig4_scene_coverage.png', bbox_inches='tight')
plt.show()
print('Saved → docs/figures/fig4_scene_coverage.png')

## 5. Conclusions

In [ ]:
from IPython.display import display, Markdown

# ── computed from df ──────────────────────────────────────────────
n_total      = len(df)
cat_counts   = df['category'].value_counts().to_dict()
n_synthetic  = (df['is_synthetic'] == 'yes').sum()
n_real       = n_total - n_synthetic

wc_min  = int(df['word_count'].min())
wc_max  = int(df['word_count'].max())
wc_mean = df['word_count'].mean()

WORDS_PER_SCENE_LOW  = 25
WORDS_PER_SCENE_HIGH = 35
scenes_low  = int((wc_min / WORDS_PER_SCENE_HIGH))
scenes_high = int(round(wc_max / WORDS_PER_SCENE_LOW))
longest_doc = df.loc[df['word_count'].idxmax(), 'file_name'].replace('.txt','')
longest_scenes = int(round(df['word_count'].max() / WORDS_PER_SCENE_LOW))
short_docs = df[df['word_count'] < 90]['file_name'].tolist()

sent_by_cat = df.groupby('category')['avg_words_per_sentence'].mean()

df['vocab_ratio'] = df['unique_words'] / df['word_count'] * 100
vr_min = df['vocab_ratio'].min()
vr_max = df['vocab_ratio'].max()

in_target = ((df['scenes_low'] >= 5) & (df['scenes_high'] <= 8)).sum() \
    if 'scenes_low' in df.columns else '?'

# ── category breakdown string ──────────────────────────────────────
cat_str = ', '.join(f"{c} ({cat_counts.get(c,'?')})" for c in ['education','product','tutorial'])

# ── sentence complexity rows ───────────────────────────────────────
sent_rows = '\n'.join(
    f'| {c.capitalize()} | **{sent_by_cat.get(c, 0):.1f} words** |'
    for c in ['education','tutorial','product'] if c in sent_by_cat
)

# ── short doc warning ─────────────────────────────────────────────
short_warning = ''
if short_docs:
    short_warning = (
        f' Very short texts ({len(short_docs)} doc(s): '
        + ', '.join(f'`{d}`' for d in short_docs)
        + ') risk producing fewer than 5 scenes; the LLM prompt should enforce a minimum.'
    )

md = f"""
### What the data tells us about the pipeline

**Corpus overview.**
The dataset contains **{n_total} texts** across three categories — {cat_str} — all in English.
{n_synthetic} document(s) are AI-generated; the remaining {n_real} come from real web sources.

**Length and scene generation.**
Word counts range from **{wc_min} to {wc_max}**, with a mean of ~**{wc_mean:.1f} words**.
At a rate of {WORDS_PER_SCENE_LOW}–{WORDS_PER_SCENE_HIGH} words per narrated scene, texts can yield
between {scenes_low} and {scenes_high} scenes. {in_target}/10 documents fall naturally within the
pipeline target of **5–8 scenes**. Longer documents (e.g., `{longest_doc}`) can yield up to
{longest_scenes} scenes and may need a *merge-short-scenes* post-processing step.{short_warning}

**Text complexity.**
Average sentence length by category:

| Category | Avg words/sentence |
|---|---|
{sent_rows}

Shorter, more direct sentences ease TTS naturalness and subtitle synchronisation.
Education texts may need sentence-splitting before being passed to the TTS engine.

**Vocabulary richness.**
Unique-word ratio ranges from **{vr_min:.0f}–{vr_max:.0f} %** of total words, indicating moderate
lexical diversity — well-suited for keyword extraction (used in the image-search step).
No text shows near-zero diversity, so keyword deduplication logic is not critical at this scale.

**Pipeline implications.**

| Observation | Pipeline action |
|---|---|
| Tutorials use bullet/numbered lists | LLM prompt should handle structured text, not only prose |
| Product texts are shortest (mean {df[df.category=='product']['word_count'].mean():.0f} words) | Padding / expansion prompt may be needed to reach 5 scenes |
| Education texts have longest sentences ({sent_by_cat.get('education', 0):.1f} words/sent) | Pre-split sentences before TTS call |
| {n_synthetic/n_total*100:.0f} % AI-generated | Benchmark TTS quality on both real and synthetic texts |
"""

display(Markdown(md))
